# Trabalho Final de IA — Steel Plates Faults
**Aluno:** Renan Dias | **Papel:** B — Engenheiro de processo | **Data:** 11/10/2026

**Decisão apoiada:** ajustar ou não o parâmetro de laminação.

**Alvo binário (definição operacional):**
- `1` = defeito cuja causa provável é **ajustável por parâmetro de laminação** (Bumps, Z_Scratch, K_Scatch);
- `0` = defeito que **ajustar o parâmetro não resolve** (Pastry, Stains, Dirtiness, Other_Faults).

**Como usar:** `Kernel → Restart Kernel and Run All`. Todo número dos slides precisa existir nas saídas deste notebook.

## 1. Carga e validação da base

In [21]:
from pathlib import Path          # Path: caminhos de arquivo portáteis (sempre relativos neste projeto)
import json                       # json: usado no fim para embutir os dados no HTML
import joblib                     # joblib: salva e carrega o modelo em disco
import pandas as pd               # pandas: tabelas (DataFrame)
import sklearn                    # sklearn: aqui só para registrar a versão usada
from sklearn.dummy import DummyClassifier                                   # "piso": chuta a classe mais comum
from sklearn.tree import DecisionTreeClassifier, export_text                # árvore de decisão e leitura em texto
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict   # divisão, busca, validação cruzada

RAIZ = Path(".")                                          # raiz do projeto (o notebook roda dentro da pasta do projeto)
df = pd.read_csv(RAIZ / "data" / "steel_plates.csv")      # lê a base montada pelo montar_csv.py
assert df.shape == (1941, 34), "a base mudou: esperado 1941 linhas x 34 colunas"   # contrato: formato esperado

print("shape:", df.shape)                                 # linhas e colunas
print("ausentes:", int(df.isna().sum().sum()))            # total de valores ausentes (esperado 0)
print("duplicadas:", int(df.duplicated().sum()))          # total de linhas repetidas (esperado 0)

shape: (1941, 34)
ausentes: 0
duplicadas: 0


## 2. O alvo binário (decisão de negócio)

In [22]:
DEFEITOS = ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]   # os 7 indicadores de defeito
PROCESSO = ["Bumps", "Z_Scratch", "K_Scatch"]             # defeitos ajustáveis por parâmetro de laminação -> alvo = 1
ROTULOS = {1: "ajustável por parâmetro", 0: "não ajustável por parâmetro"}   # nomes das classes

df["alvo"] = df[PROCESSO].sum(axis=1).astype(int)         # alvo = 1 se a placa tem um dos defeitos de PROCESSO
assert df["alvo"].isin([0, 1]).all()                      # contrato: o alvo é binário

print(df["alvo"].value_counts())                          # contagem por classe (esperado: 1 -> 983, 0 -> 958)
print((df["alvo"].value_counts(normalize=True) * 100).round(1))   # proporção em %
print(pd.crosstab(df[DEFEITOS].idxmax(axis=1), df["alvo"]))       # cada defeito deve cair inteiro em uma só classe

alvo
1    983
0    958
Name: count, dtype: int64
alvo
1    50.6
0    49.4
Name: proportion, dtype: float64
alvo            0    1
row_0                 
Bumps           0  402
Dirtiness      55    0
K_Scatch        0  391
Other_Faults  673    0
Pastry        158    0
Stains         72    0
Z_Scratch       0  190


## 3. Entradas (X), alvo (y) e defeito original (`meta`) — sem vazamento
As 7 colunas de defeito **são** o alvo disfarçado: saem do X. O `meta` guarda o defeito original só para a análise de erro.

In [24]:
meta = df[DEFEITOS].idxmax(axis=1).rename("defeito_original")   # nome do defeito de cada placa (só para análise de erro)
X = df.drop(columns=DEFEITOS + ["alvo"])                  # entradas: as 27 variáveis de sensor
y = df["alvo"]                                            # o que o modelo precisa prever (0 ou 1)

assert X.shape == (1941, 27)                              # contrato 1: 1941 linhas x 27 colunas
assert not set(DEFEITOS) & set(X.columns), "vazamento: coluna de defeito em X"   # contrato 2: sem vazamento

## 4. Divisão treino/teste (fixa, conforme o enunciado)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(     # divide as linhas em treino (80%) e teste (20%)
    X, y, test_size=0.20, random_state=42                 # random_state fixo: a divisão é sempre a mesma
)
meta_train, meta_test = meta.loc[X_train.index], meta.loc[X_test.index]   # defeito original alinhado pelo índice

print(X_train.shape, X_test.shape)                        # esperado: (1552, 27) (389, 27)
print("proporção da classe 1 -> treino:", round(float(y_train.mean()), 3), "| teste:", round(float(y_test.mean()), 3))

(1552, 27) (389, 27)
proporção da classe 1 -> treino: 0.507 | teste: 0.504


## 5. Exploração (somente no treino)

In [26]:
print("dimensões:", X_train.shape)                        # linhas e colunas do treino
print(y_train.value_counts())                             # quantas placas em cada classe no treino
print((y_train.value_counts(normalize=True) * 100).round(1))   # proporção em %
print("ausentes:", int(X_train.isna().sum().sum()))       # valores ausentes no treino
print("duplicadas:", int(X_train.duplicated().sum()))     # linhas repetidas no treino
print(meta_train.value_counts())                          # defeitos originais no treino
print(X_train.describe().T[["min", "mean", "max"]])       # mínimo, média e máximo por variável (escalas bem diferentes)
print(X_train[["TypeOfSteel_A300", "TypeOfSteel_A400"]].sum(axis=1).value_counts())   # soma sempre 1 => colunas complementares

dimensões: (1552, 27)
alvo
1    787
0    765
Name: count, dtype: int64
alvo
1    50.7
0    49.3
Name: proportion, dtype: float64
ausentes: 0
duplicadas: 0
defeito_original
Other_Faults    530
Bumps           330
K_Scatch        308
Z_Scratch       149
Pastry          129
Stains           59
Dirtiness        47
Name: count, dtype: int64
                             min          mean           max
X_Minimum                 0.0000  5.806894e+02  1.705000e+03
X_Maximum                 4.0000  6.281160e+02  1.713000e+03
Y_Minimum              6712.0000  1.661509e+06  1.298766e+07
Y_Maximum              6724.0000  1.661566e+06  1.298769e+07
Pixels_Areas              2.0000  1.915972e+03  1.526550e+05
X_Perimeter               2.0000  1.138724e+02  1.044900e+04
Y_Perimeter               1.0000  8.552964e+01  1.815200e+04
Sum_of_Luminosity       255.0000  2.086904e+05  1.159141e+07
Minimum_of_Luminosity     0.0000  8.490399e+01  1.960000e+02
Maximum_of_Luminosity    37.0000  1.302771e+02  2.53

**Decisão de preparação:** sem ausentes, sem categóricas a codificar, e árvore de decisão não depende da escala das variáveis.
Nenhuma transformação por ora. Se um terceiro modelo sensível à escala entrar, o `StandardScaler` vai num `Pipeline`, ajustado **só no treino**.
`TypeOfSteel_A300` e `TypeOfSteel_A400` são complementares (somam 1): inofensivo para árvore; para modelo linear seria preciso remover uma.

## 6. Modelos: piso, regra simples e árvore

In [27]:
piso = DummyClassifier(strategy="most_frequent")          # piso: sempre prevê a classe mais frequente do treino
piso.fit(X_train, y_train)                                # fit = TREINO (usa só X_train e y_train)

regra = DecisionTreeClassifier(max_depth=1, random_state=42)   # regra simples: um único corte em uma única variável
regra.fit(X_train, y_train)                               # treina a regra só com o treino

arvore_livre = DecisionTreeClassifier(random_state=42)    # árvore sem limite: cresce até decorar o treino
arvore_livre.fit(X_train, y_train)                        # treina a árvore sem poda

print(export_text(regra, feature_names=list(X_train.columns)))   # a regra simples escrita em texto legível

|--- Pixels_Areas <= 3606.50
|   |--- class: 0
|--- Pixels_Areas >  3606.50
|   |--- class: 1



## 7. Poda por validação cruzada (só no treino — nunca olhar o teste para escolher)

In [28]:
grade = {"max_depth": [2, 3, 4, 5, 6, 8, None],           # profundidades testadas (None = sem limite)
         "min_samples_leaf": [1, 5, 10, 20]}              # tamanho mínimo de cada folha
busca = GridSearchCV(DecisionTreeClassifier(random_state=42), grade,
                     cv=5, scoring="roc_auc")             # 5 partes dentro do treino; nota = área sob a curva ROC
busca.fit(X_train, y_train)                               # roda a busca usando só o treino
print(busca.best_params_, round(busca.best_score_, 3))    # combinação vencedora e sua nota média
arvore_podada = busca.best_estimator_                     # árvore vencedora, já retreinada no treino inteiro

for nome, valores in grade.items():                       # checa se o vencedor está na BORDA da grade
    vencedor = busca.best_params_[nome]                   # valor vencedor deste hiperparâmetro
    numericos = [v for v in valores if v is not None]     # valores numéricos da lista
    na_borda = vencedor is None or vencedor in (min(numericos), max(numericos))   # menor, maior ou "sem limite"
    print(f"{nome}: vencedor={vencedor} | na borda da grade? {na_borda}")   # se True, a grade pode ser estendida (decisão sobre a validação, não sobre o teste)

{'max_depth': None, 'min_samples_leaf': 10} 0.849
max_depth: vencedor=None | na borda da grade? True
min_samples_leaf: vencedor=10 | na borda da grade? False


## 8. Treino × teste (acurácia) e diagnóstico de sobreajuste

In [29]:
modelos = {"piso (classe majoritária)": piso, "regra simples (1 corte)": regra,
           "árvore sem poda": arvore_livre, "árvore podada": arvore_podada}   # nome -> modelo
for nome, m in modelos.items():                           # percorre os modelos
    acc_tr = m.score(X_train, y_train)                    # acurácia no treino
    acc_te = m.score(X_test, y_test)                      # acurácia no teste (dados nunca vistos)
    print(f"{nome:28s} treino={acc_tr:.3f}  teste={acc_te:.3f}  diferença={acc_tr - acc_te:+.3f}")

piso (classe majoritária)    treino=0.507  teste=0.504  diferença=+0.003
regra simples (1 corte)      treino=0.644  teste=0.645  diferença=-0.002
árvore sem poda              treino=1.000  teste=0.802  diferença=+0.198
árvore podada                treino=0.891  teste=0.787  diferença=+0.104


In [30]:
for nome, m in [("sem poda", arvore_livre), ("podada", arvore_podada)]:   # compara as probabilidades das duas árvores
    p = m.predict_proba(X_test)[:, 1]                     # probabilidade da classe 1 para cada placa de teste
    print(nome, "-> valores distintos de probabilidade:", len(set(p.round(3))))   # quantos valores diferentes
    print(nome, "-> placas entre 0,2 e 0,8:", int(((p > 0.2) & (p < 0.8)).sum()))  # quantas placas "indecisas"

sem poda -> valores distintos de probabilidade: 2
sem poda -> placas entre 0,2 e 0,8: 0
podada -> valores distintos de probabilidade: 36
podada -> placas entre 0,2 e 0,8: 95


A árvore sem poda só devolve 0 ou 1 (folhas puras): um controle deslizante de corte não teria efeito e não haveria faixa de revisão.
A árvore podada devolve probabilidades graduais — por isso ela é o modelo do projeto, independentemente de pequenas diferenças de acurácia no teste.

## 9. Premissas de negócio — 
- `R = custo(falso negativo) / custo(falso positivo)`: quanto pior é **não ajustar quando era ajustável** (defeito continua sendo produzido) do que **ajustar sem necessidade**.
- `LARGURA`: metade da largura da faixa de revisão humana em torno do corte (revisão = corte ± LARGURA).
Estes dois números são **premissas suas**, não dados da base. Justifique-os com a sua experiência de planta.

In [33]:
R = 3.0                                                   # PREMISSA : FN custa 3x mais que FP
LARGURA = 0.10                                            # PREMISSA : faixa de revisão = corte ± 0,10
GRADE_CORTES = [i / 100 for i in range(1, 100)]           # cortes candidatos: 0.01, 0.02, ..., 0.99


def contar(y_real, p, corte):
    """Conta VP, FP, FN, VN: a previsão é 1 quando p >= corte."""
    pred = p >= corte                                     # vetor True/False: o modelo diz "ajustável" neste corte?
    real = y_real.to_numpy() == 1                         # vetor True/False: a placa é realmente ajustável?
    vp = int((pred & real).sum())                         # verdadeiro positivo: previu 1 e era 1
    fp = int((pred & ~real).sum())                        # falso positivo: previu 1 e era 0
    fn = int((~pred & real).sum())                        # falso negativo: previu 0 e era 1
    vn = int((~pred & ~real).sum())                       # verdadeiro negativo: previu 0 e era 0
    return vp, fp, fn, vn


def custo(y_real, p, corte, r):
    """Custo total = r * FN + 1 * FP (o falso positivo vale 1 unidade)."""
    vp, fp, fn, vn = contar(y_real, p, corte)             # contagens no corte
    return r * fn + fp                                    # custo ponderado dos dois erros


def melhor_corte(y_real, p, r):
    """Corte da grade que minimiza o custo; no empate, o mais próximo de 0,5."""
    return min(GRADE_CORTES, key=lambda t: (custo(y_real, p, t, r), abs(t - 0.5)))

## 10. Ponto de corte pelo custo do erro (escolhido no TREINO, por validação cruzada)

In [34]:
p_oof = cross_val_predict(arvore_podada, X_train, y_train, cv=5, method="predict_proba")[:, 1]   # probabilidades "fora da amostra" no treino

linhas = []                                               # tabela de sensibilidade ao custo R
for r in [1, 2, 3, 5, 10]:                                # valores de R testados
    t = melhor_corte(y_train, p_oof, r)                   # corte ótimo para este R
    vp, fp, fn, vn = contar(y_train, p_oof, t)            # erros nesse corte
    linhas.append({"R": r, "corte_ótimo": t, "teórico_1/(1+R)": round(1 / (1 + r), 2), "FN": fn, "FP": fp, "custo": r * fn + fp})
print(pd.DataFrame(linhas).to_string(index=False))        # quanto o corte se move quando o custo relativo muda

T_ESCOLHIDO = melhor_corte(y_train, p_oof, R)             # corte final, com o R definido nas premissas
print("R =", R, "| corte escolhido =", T_ESCOLHIDO)

 R  corte_ótimo  teórico_1/(1+R)  FN  FP  custo
 1         0.46             0.50 160 178    338
 2         0.35             0.33 130 230    490
 3         0.05             0.25  51 408    561
 5         0.05             0.17  51 408    663
10         0.05             0.09  51 408    918
R = 3.0 | corte escolhido = 0.05


**Leitura:** com probabilidades bem calibradas, o corte ótimo é `1/(1+R)` (coluna "teórico"). Quanto maior o custo do falso negativo (R), mais **baixo** o corte:
o sistema aceita mais alarmes falsos para não deixar passar defeito ajustável. O corte do notebook fica perto do teórico, mas não igual, porque as probabilidades de uma árvore não são perfeitamente calibradas.

## 11. Avaliação no corte escolhido — treino × teste

In [35]:
p_train = arvore_podada.predict_proba(X_train)[:, 1]      # probabilidades no treino (o modelo já viu: visão otimista)
p_test = arvore_podada.predict_proba(X_test)[:, 1]        # probabilidades no teste (dados nunca vistos)


def metricas(y_real, p, corte):
    """Acurácia, precisão, recall e a matriz de confusão num corte."""
    vp, fp, fn, vn = contar(y_real, p, corte)             # contagens
    total = vp + fp + fn + vn                             # nº de placas
    return {"acuracia": (vp + vn) / total,                # fração de acertos
            "precisao": vp / (vp + fp) if (vp + fp) else 0.0,   # dos que o modelo mandou ajustar, quantos eram ajustáveis
            "recall": vp / (vp + fn) if (vp + fn) else 0.0,     # dos ajustáveis, quantos o modelo pegou
            "VP": vp, "FP": fp, "FN": fn, "VN": vn}


m_train = metricas(y_train, p_train, T_ESCOLHIDO)         # métricas no treino
m_test = metricas(y_test, p_test, T_ESCOLHIDO)            # métricas no teste
print(pd.DataFrame({"treino": m_train, "teste": m_test}).round(3))   # tabela lado a lado

LIMITE_SOBREAJUSTE = 0.05                                 # premissa da SKILL: lacuna acima de 5 pontos = sobreajuste relevante
for k in ["acuracia", "precisao", "recall"]:              # diagnóstico por métrica
    lacuna = m_train[k] - m_test[k]                       # diferença treino - teste
    veredito = "SOBREAJUSTE RELEVANTE" if lacuna > LIMITE_SOBREAJUSTE else "ok"   # compara com o limite
    print(f"{k}: treino={m_train[k]:.3f} teste={m_test[k]:.3f} lacuna={lacuna:+.3f} -> {veredito}")

print("\nMatriz de confusão no TESTE (linhas = real, colunas = previsto):")
print(pd.DataFrame([[m_test["VN"], m_test["FP"]], [m_test["FN"], m_test["VP"]]],
                   index=["real 0 (não ajustável)", "real 1 (ajustável)"],
                   columns=["previsto 0", "previsto 1"]))   # matriz de confusão no teste

           treino    teste
acuracia    0.758    0.730
precisao    0.677    0.659
recall      1.000    0.964
VP        787.000  189.000
FP        376.000   98.000
FN          0.000    7.000
VN        389.000   95.000
acuracia: treino=0.758 teste=0.730 lacuna=+0.028 -> ok
precisao: treino=0.677 teste=0.659 lacuna=+0.018 -> ok
recall: treino=1.000 teste=0.964 lacuna=+0.036 -> ok

Matriz de confusão no TESTE (linhas = real, colunas = previsto):
                        previsto 0  previsto 1
real 0 (não ajustável)          95          98
real 1 (ajustável)               7         189


## 12. O modelo é melhor que a prática atual? (pergunta 2, em termos operacionais)
Referência = piso e regra de 1 corte (proxy de "olhar um indicador"; premissa). Custo por 100 placas = `(R·FN + FP) / N · 100`.

In [37]:
def linha_comparacao(nome, y_real, pred_ou_p, corte):
    """Resume um modelo em erros por 100 placas e custo por 100 placas."""
    vp, fp, fn, vn = contar(y_real, pred_ou_p, corte)     # contagens
    n = vp + fp + fn + vn                                 # total de placas
    return {"modelo": nome, "FN por 100": round(fn / n * 100, 1), "FP por 100": round(fp / n * 100, 1),
            "erros por 100": round((fn + fp) / n * 100, 1), "custo por 100": round((R * fn + fp) / n * 100, 1)}


comparacao = pd.DataFrame([
    linha_comparacao("piso", y_test, piso.predict(X_test), 0.5),                  # sempre a classe majoritária
    linha_comparacao("regra simples (1 corte)", y_test, regra.predict(X_test), 0.5),   # previsão 0/1 da regra
    linha_comparacao("árvore podada no corte escolhido", y_test, p_test, T_ESCOLHIDO),  # modelo final, sem faixa
])
print(comparacao.to_string(index=False))

                          modelo  FN por 100  FP por 100  erros por 100  custo por 100
                            piso         0.0        49.6           49.6           49.6
         regra simples (1 corte)        34.2         1.3           35.5          103.9
árvore podada no corte escolhido         1.8        25.2           27.0           30.6


## 13. Faixa de revisão humana (pergunta 4)
`p >= corte + LARGURA` → ajustar automaticamente · `p < corte − LARGURA` → não ajustar automaticamente · no meio → **revisão humana**.
Os erros automáticos são os que sobram depois da revisão; a revisão é o trabalho humano gerado.

In [38]:
def faixa(y_real, p, corte, largura):
    """Conta decisões automáticas, erros automáticos e casos para revisão humana."""
    real = y_real.to_numpy() == 1                         # a placa é realmente ajustável?
    auto_aj = p >= corte + largura                        # ajustar automaticamente
    auto_nao = p < corte - largura                        # não ajustar automaticamente
    rev = ~(auto_aj | auto_nao)                           # todo o resto vai para um humano
    n = len(p)                                            # nº de placas
    fp_auto = int((auto_aj & ~real).sum())                # ajustou sem necessidade (automático)
    fn_auto = int((auto_nao & real).sum())                # deixou de ajustar o que era ajustável (automático)
    return {"revisão por 100": round(float(rev.sum()) / n * 100, 1), "auto: ajustar": int(auto_aj.sum()),
            "auto: não ajustar": int(auto_nao.sum()), "revisão": int(rev.sum()),
            "FP automático": fp_auto, "FN automático": fn_auto,
            "custo por 100 (revisão perfeita)": round((R * fn_auto + fp_auto) / n * 100, 1)}


resumo_faixa = faixa(y_test, p_test, T_ESCOLHIDO, LARGURA)   # resultado no teste com a largura escolhida
print(resumo_faixa)                                       # inclui: casos para revisão a cada 100

sens = []                                                 # sensibilidade à largura da faixa
for w in [0.0, 0.05, 0.10, 0.15, 0.20]:                   # larguras testadas
    s = faixa(y_test, p_test, T_ESCOLHIDO, w)             # resultado para cada largura
    sens.append({"largura": w, "revisão por 100": s["revisão por 100"], "FP automático": s["FP automático"],
                 "FN automático": s["FN automático"], "custo por 100": s["custo por 100 (revisão perfeita)"]})
print(pd.DataFrame(sens).to_string(index=False))          # trade-off: mais revisão humana x menos erro automático

{'revisão por 100': 33.7, 'auto: ajustar': 258, 'auto: não ajustar': 0, 'revisão': 131, 'FP automático': 73, 'FN automático': 0, 'custo por 100 (revisão perfeita)': 18.8}
 largura  revisão por 100  FP automático  FN automático  custo por 100
    0.00              0.0             98              7           30.6
    0.05             29.0             88              0           22.6
    0.10             33.7             73              0           18.8
    0.15             34.7             70              0           18.0
    0.20             39.6             60              0           15.4


## 14. Análise de erro por defeito original (SKILL) — onde o modelo erra

In [39]:
analise = pd.DataFrame({"p": p_test, "real": y_test.to_numpy()}, index=X_test.index).join(meta_test)   # prob., classe real e defeito por placa
analise["previsto"] = (analise["p"] >= T_ESCOLHIDO).astype(int)                   # previsão no corte
analise["revisao"] = (analise["p"] >= T_ESCOLHIDO - LARGURA) & (analise["p"] < T_ESCOLHIDO + LARGURA)   # caiu na faixa de revisão?
por_defeito = analise.groupby("defeito_original").agg(
    n=("previsto", "size"),                               # nº de placas deste defeito no teste
    classe_esperada=("real", "first"),                    # classe que o alvo define para este defeito
    pct_previsto_ajustar=("previsto", "mean"),            # % que o modelo manda ajustar
    pct_erro=("previsto", lambda s: float((s != analise.loc[s.index, "real"]).mean())),   # % de erro no corte
    pct_revisao=("revisao", "mean"),                      # % que cai na faixa de revisão
).round(3).sort_values("n", ascending=False)
print(por_defeito)                                        # Other_Faults é o caso a observar; Dirtiness e Stains têm poucas placas (leitura frágil)

                    n  classe_esperada  pct_previsto_ajustar  pct_erro  \
defeito_original                                                         
Other_Faults      143                0                 0.573     0.573   
K_Scatch           83                1                 1.000     0.000   
Bumps              72                1                 0.917     0.083   
Z_Scratch          41                1                 0.976     0.024   
Pastry             29                0                 0.345     0.345   
Stains             13                0                 0.077     0.077   
Dirtiness           8                0                 0.625     0.625   

                  pct_revisao  
defeito_original               
Other_Faults            0.559  
K_Scatch                0.024  
Bumps                   0.097  
Z_Scratch               0.049  
Pastry                  0.690  
Stains                  1.000  
Dirtiness               0.875  


## 15. Persistência (joblib) e função `prever`

In [40]:
pacote = {                                                # tudo que o demo.py precisa, sem retreinar nada
    "modelo": arvore_podada,                              # árvore já treinada
    "colunas": list(X.columns),                           # ordem das 27 variáveis
    "corte": float(T_ESCOLHIDO),                          # ponto de corte escolhido pelo custo do erro
    "largura_faixa": float(LARGURA),                      # metade da faixa de revisão humana
    "faixas": {c: (float(X_train[c].min()), float(X_train[c].max())) for c in X.columns},   # faixa vista no treino
    "rotulos": ROTULOS,                                   # textos das classes
    "defeitos": DEFEITOS, "processo": PROCESSO,           # definição do alvo (fonte única da verdade)
    "R": R,                                               # custo relativo adotado
    "versoes": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__},   # versões usadas no treino
}
joblib.dump(pacote, RAIZ / "modelo.joblib", compress=3)   # salva em disco, comprimido
recarregado = joblib.load(RAIZ / "modelo.joblib")         # recarrega do disco
iguais = bool((recarregado["modelo"].predict_proba(X_test)[:, 1] == p_test).all())   # as probabilidades são idênticas?
print("modelo recarregado igual ao original:", iguais, "| tamanho:", (RAIZ / "modelo.joblib").stat().st_size, "bytes")

modelo recarregado igual ao original: True | tamanho: 5414 bytes


In [41]:
from demo import prever                                   # a função oficial está em demo.py (carrega o modelo do disco)

exemplo = X_test.iloc[0].to_dict()                        # um caso real de teste, como dicionário
print("real:", int(y_test.iloc[0]), "->", prever(exemplo))   # classe, probabilidade, revisão humana

real: 0 -> {'classe': 1, 'rotulo': 'ajustável por parâmetro', 'probabilidade': 0.2778, 'revisao_humana': False, 'motivo_revisao': None, 'avisos': []}


## 16. Modo de falha provocado: **falha silenciosa**
Uma versão de `prever` **sem validação** devolve uma resposta de aparência normal mesmo para entrada absurda. O resultado parece válido, e ninguém percebe.
Mitigação: validação de entrada (erro explícito) e entrada fora da faixa de treino enviada para revisão humana.

In [42]:
def prever_sem_validacao(registro):
    """VERSÃO RUIM de propósito: não valida nada."""
    b = joblib.load(RAIZ / "modelo.joblib")               # carrega o modelo
    linha = pd.DataFrame([registro])[b["colunas"]]        # monta a linha sem conferir valores
    p = float(b["modelo"].predict_proba(linha)[0, 1])     # prevê qualquer coisa
    return {"probabilidade": round(p, 4), "classe": int(p >= b["corte"])}   # resposta de aparência normal


normal = X_test.iloc[0].to_dict()                         # registro válido
absurdo = {**normal, "Pixels_Areas": -999999}             # valor impossível (área negativa)
com_nan = {**normal, "Pixels_Areas": float("nan")}        # valor ausente

for nome, entrada in [("válido", normal), ("área absurda", absurdo), ("valor NaN", com_nan)]:   # roda os três casos
    try:
        print(f"SEM validação | {nome:13s} ->", prever_sem_validacao(entrada))   # resposta silenciosa
    except Exception as e:                                # em versões antigas do scikit-learn o NaN gera erro
        print(f"SEM validação | {nome:13s} -> erro do scikit-learn: {e}")
    try:
        print(f"COM validação | {nome:13s} ->", prever(entrada))   # mitigação aplicada
    except ValueError as e:                               # entrada inválida é recusada com mensagem clara
        print(f"COM validação | {nome:13s} -> recusada: {e}")

SEM validação | válido        -> {'probabilidade': 0.2778, 'classe': 1}
COM validação | válido        -> {'classe': 1, 'rotulo': 'ajustável por parâmetro', 'probabilidade': 0.2778, 'revisao_humana': False, 'motivo_revisao': None, 'avisos': []}
SEM validação | área absurda  -> {'probabilidade': 0.2778, 'classe': 1}
COM validação | área absurda  -> {'classe': 1, 'rotulo': 'ajustável por parâmetro', 'probabilidade': 0.2778, 'revisao_humana': True, 'motivo_revisao': 'entrada fora da faixa de treino', 'avisos': ['Pixels_Areas=-999999 fora da faixa de treino [2.0, 152655.0]']}
SEM validação | valor NaN     -> {'probabilidade': 0.2778, 'classe': 1}
COM validação | valor NaN     -> recusada: valor invalido em Pixels_Areas: nan (esperado numero finito)


## 17. Interface local (`interface.html`) com dados embutidos

In [43]:
dados = {
    "rotulos": {"1": ROTULOS[1], "0": ROTULOS[0]},        # nomes das classes
    "corte": float(T_ESCOLHIDO), "largura": float(LARGURA), "R": float(R),   # parâmetros do notebook
    "metricas": {"treino": {k: float(v) for k, v in m_train.items()},          # métricas do notebook (treino)
                 "teste": {k: float(v) for k, v in m_test.items()}},           # métricas do notebook (teste)
    "casos": [{"p": round(float(p), 4), "real": int(r), "defeito": str(d)}     # um registro por placa de teste
              for p, r, d in zip(p_test, y_test, meta_test)],
}
modelo_html = (RAIZ / "modelo_interface.template.html").read_text(encoding="utf-8")   # lê o modelo de página
dados_json = json.dumps(dados, ensure_ascii=False).replace("</", "<\\/")      # JSON seguro para dentro de <script>
pagina = modelo_html.replace("__DADOS__", dados_json)     # embute os dados na página
assert "__DADOS__" not in pagina                          # contrato: nada ficou sem substituir
(RAIZ / "interface.html").write_text(pagina, encoding="utf-8")   # grava o arquivo final
print("interface.html gerado:", (RAIZ / "interface.html").stat().st_size, "bytes | casos:", len(dados["casos"]))

interface.html gerado: 30427 bytes | casos: 389


## 18. Resumo dos números para os slides (todos vêm das células acima)

In [44]:
print("Placas por classe (base inteira):", df["alvo"].value_counts().to_dict())          # slide 3
print("Treino/teste:", X_train.shape[0], "/", X_test.shape[0])                            # slide 2
print("Acurácia no teste (piso, regra, livre, podada):",
      [round(m.score(X_test, y_test), 3) for m in modelos.values()])                       # slide 4
print("R =", R, "| corte =", T_ESCOLHIDO, "| largura da faixa =", LARGURA)                 # slide 6
print("Teste no corte:", {k: round(v, 3) for k, v in m_test.items()})                      # slide 5
print("Comparação operacional:\n", comparacao.to_string(index=False))                      # slide 7
print("Faixa de revisão:", resumo_faixa)                                                   # slide 9

Placas por classe (base inteira): {1: 983, 0: 958}
Treino/teste: 1552 / 389
Acurácia no teste (piso, regra, livre, podada): [0.504, 0.645, 0.802, 0.787]
R = 3.0 | corte = 0.05 | largura da faixa = 0.1
Teste no corte: {'acuracia': 0.73, 'precisao': 0.659, 'recall': 0.964, 'VP': 189, 'FP': 98, 'FN': 7, 'VN': 95}
Comparação operacional:
                           modelo  FN por 100  FP por 100  erros por 100  custo por 100
                            piso         0.0        49.6           49.6           49.6
         regra simples (1 corte)        34.2         1.3           35.5          103.9
árvore podada no corte escolhido         1.8        25.2           27.0           30.6
Faixa de revisão: {'revisão por 100': 33.7, 'auto: ajustar': 258, 'auto: não ajustar': 0, 'revisão': 131, 'FP automático': 73, 'FN automático': 0, 'custo por 100 (revisão perfeita)': 18.8}
